## TPC (MIMIC-IV remaining LoS) end-to-end debug notebook

This notebook validates the implementation **step-by-step** with **deterministic, checkable expected values**.

It has two parts:

- **Part A (deterministic synthetic)**: builds a tiny synthetic sample where the expected values (e.g. decay indicators) are *exact*, and asserts them.
- **Part B (optional real data)**: runs the full `MIMIC4EHRDataset` + `RemainingLengthOfStayTPC_MIMIC4` pipeline on the included MIMIC-IV demo data (outputs are not deterministic, but the shapes and keys are).


In [ ]:
# Part A — deterministic synthetic verification

import os
import sys
from datetime import datetime, timedelta

# Ensure we import the *local repo* version of pyhealth
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "examples" else os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Put cache inside repo (avoids permission issues in sandboxes)
os.environ.setdefault("PYHEALTH_CACHE_PATH", os.path.join(REPO_ROOT, ".pyhealth_cache"))

import torch
from pyhealth.datasets import create_sample_dataset
from pyhealth.datasets.utils import collate_fn_dict_with_padding
from pyhealth.models import TPC

print("Repo root:", REPO_ROOT)
print("PYHEALTH_CACHE_PATH:", os.environ.get("PYHEALTH_CACHE_PATH"))
print("Torch:", torch.__version__)

torch.manual_seed(0)


### A1) Build a tiny raw `ts` payload (what the Task produces)

We use a **2-feature** setup with itemids `A` and `B`.

- `prefill_start = 2020-01-01 00:00`
- `pred_start   = 2020-01-01 05:00`
- `pred_end     = 2020-01-01 10:00`  → **T = 5** prediction hours (5,6,7,8,9)

Observations:
- Feature **A** observed at hour 0 with value 10.
- Feature **B** observed at hour 2 with value 1.

**Expected decay at the first prediction step (hour 5):**
- For **A**: last seen at hour 0, so \(\Delta t = 5\) → decay = \(0.75^5 = 0.2373046875\)
- For **B**: last seen at hour 2, so \(\Delta t = 3\) → decay = \(0.75^3 = 0.421875\)


In [ ]:
from pprint import pprint

prefill_start = datetime(2020, 1, 1, 0, 0, 0)
pred_start = prefill_start + timedelta(hours=5)
pred_end = prefill_start + timedelta(hours=10)

raw_ts = {
    "prefill_start": prefill_start,
    "icu_start": prefill_start,
    "pred_start": pred_start,
    "pred_end": pred_end,
    "feature_itemids": ["A", "B"],
    "long_df": {
        "timestamp": [
            prefill_start + timedelta(hours=0),
            prefill_start + timedelta(hours=2),
        ],
        "itemid": ["A", "B"],
        "value": [10.0, 1.0],
        "source": ["chartevents", "chartevents"],
    },
}

pprint(raw_ts)


### A2) Create an in-memory `SampleDataset` and inspect processor outputs

We build two samples so batching works, then check:

- `dataset[0]["ts"]` has shape **(T=5, F=2, C=2)**.
- The decay indicators match the expected values at `t=0` (first prediction hour).

Because the time-series processor does robust scaling using 5th/95th percentiles from the dataset, the **scaled values** depend on the sample distribution; we therefore verify the **decay** exactly (it is independent of scaling) and only sanity-check the value channel.


In [ ]:
samples = [
    {
        "patient_id": "p0",
        "stay_id": "s0",
        "hadm_id": "h0",
        "ts": raw_ts,
        "static": {
            "gender": "M",
            "race": "WHITE",
            "admission_location": "ER",
            "insurance": "Medicare",
            "first_careunit": "MICU",
            "hour_of_admission": 0,
            "admission_height": 170.0,
            "admission_weight": 80.0,
            "gcs_eye": 4.0,
            "gcs_motor": 6.0,
            "gcs_verbal": 5.0,
            "anchor_age": 65.0,
        },
        "y": [2.0, 1.5, 1.0, 0.75, 0.5],
    },
    {
        "patient_id": "p1",
        "stay_id": "s1",
        "hadm_id": "h1",
        "ts": raw_ts,
        "static": {
            "gender": "F",
            "race": "ASIAN",
            "admission_location": "CLINIC",
            "insurance": "Private",
            "first_careunit": "SICU",
            "hour_of_admission": 1,
            "admission_height": 160.0,
            "admission_weight": 55.0,
            "gcs_eye": 4.0,
            "gcs_motor": 6.0,
            "gcs_verbal": 5.0,
            "anchor_age": 50.0,
        },
        "y": [3.0, 2.0, 1.0, 0.75, 0.5],
    },
]

dataset = create_sample_dataset(
    samples=samples,
    input_schema={"ts": ("tpc_timeseries", {}), "static": ("tpc_static", {})},
    output_schema={"y": ("regression_sequence", {})},
    dataset_name="debug_tpc",
    task_name="remaining_los",
    in_memory=True,
)

x_ts = dataset[0]["ts"]
print("ts tensor shape:", tuple(x_ts.shape))

# Expected decay values at the first prediction hour
expected_decay_A = 0.75**5
expected_decay_B = 0.75**3

# layout: (T, F, 2), where channel 1 is decay
decay_A_t0 = float(x_ts[0, 0, 1].item())
decay_B_t0 = float(x_ts[0, 1, 1].item())

print("decay A@t0 =", decay_A_t0, " expected:", expected_decay_A)
print("decay B@t0 =", decay_B_t0, " expected:", expected_decay_B)

assert x_ts.shape == (5, 2, 2)
assert abs(decay_A_t0 - expected_decay_A) < 1e-6
assert abs(decay_B_t0 - expected_decay_B) < 1e-6

# Value channel sanity (not exact): forward-filled values should be non-zero after first obs.
val_A_t0 = float(x_ts[0, 0, 0].item())
val_B_t0 = float(x_ts[0, 1, 0].item())
print("value A@t0 (scaled):", val_A_t0)
print("value B@t0 (scaled):", val_B_t0)


### A3) Collate a batch (padding behavior)

We use PyHealth’s `collate_fn_dict_with_padding`.

**Expected**:
- `batch["ts"]` → shape `(B=2, T=5, F=2, C=2)`
- `batch["static"]` → shape `(B=2, S)`
- `batch["y"]` → shape `(B=2, T=5)`


In [ ]:
batch = collate_fn_dict_with_padding([dataset[0], dataset[1]])

print("batch keys:", list(batch.keys()))
print("batch ts:", tuple(batch["ts"].shape))
print("batch static:", tuple(batch["static"].shape))
print("batch y:", tuple(batch["y"].shape))

assert batch["ts"].shape == (2, 5, 2, 2)
assert batch["y"].shape == (2, 5)


### A4) Model forward + loss

We instantiate `TPC` with small hyperparameters and run a forward pass.

**Expected**:
- Output dict contains: `logit`, `y_prob`, `loss`, `y_true`
- Shapes:
  - `y_prob`: `(B=2, T=5)`
  - `logit`: `(B=2, T=5)`
- Output range:
  - `y_prob` is clipped to \([1/48, 100]\) days.


In [ ]:
model = TPC(
    dataset=dataset,
    num_layers=2,
    temporal_channels=4,
    pointwise_channels=3,
    kernel_size=3,
    main_dropout=0.0,
    temporal_dropout=0.0,
    use_batchnorm=False,  # makes tiny-batch debugging simpler
)

out = model(**batch)
print("out keys:", list(out.keys()))
print("y_prob shape:", tuple(out["y_prob"].shape))
print("logit shape:", tuple(out["logit"].shape))
print("loss:", float(out["loss"].item()))
print("y_prob min/max:", float(out["y_prob"].min().item()), float(out["y_prob"].max().item()))

assert out["y_prob"].shape == (2, 5)
assert out["logit"].shape == (2, 5)
assert torch.isfinite(out["loss"])

min_days = 1.0 / 48.0
assert float(out["y_prob"].min().item()) >= min_days - 1e-6
assert float(out["y_prob"].max().item()) <= 100.0 + 1e-6


## Part B — optional: run the full MIMIC-IV-demo pipeline

This section runs the full implementation on the included `datasets/mimic-iv-demo/2.2` demo data.

**Expected** (high-level):
- Dataset loads and caches under the repo (no permission errors).
- `dataset.set_task(...)` returns a `SampleDataset` whose items include processed tensors for:
  - `ts`: `(T, F, 2)`
  - `static`: `(S,)`
  - `y`: `(T,)`

The exact number of samples and metrics are not deterministic across environments, but the pipeline should complete without exceptions.


In [12]:
from pyhealth.datasets import MIMIC4EHRDataset, split_by_patient, get_dataloader
from pyhealth.tasks import RemainingLengthOfStayTPC_MIMIC4
from pyhealth.trainer import Trainer

import polars as pl

# Demo data path in this repo
EHR_ROOT = os.path.join(REPO_ROOT, "datasets", "mimic-iv-demo", "2.2")
CACHE_DIR = os.path.join(REPO_ROOT, ".pyhealth_dataset_cache")

# Minimal lab-only feature set (replace with paper Table 17 for faithful replication)
LAB_ITEMIDS = [
    "50824", "52455", "50983", "52623",
    "50822", "52452", "50971", "52610",
    "50806", "52434", "50902", "52535",
    "50803", "50804",
    "50809", "52027", "50931", "52569",
    "50808", "51624",
    "50960",
    "50868", "52500",
    "52031", "50964", "51701",
    "50970",
]

# -------------------------------------------------------------------
# B1) Base dataset load sanity checks (before task/processing)
# -------------------------------------------------------------------
mimic = MIMIC4EHRDataset(
    root=EHR_ROOT,
    tables=["patients", "admissions", "icustays", "labevents", "chartevents"],
    dev=True,
    num_workers=1,
    cache_dir=CACHE_DIR,
)

# Trigger event dataframe construction and verify key invariants.
gdf = mimic.global_event_df

stats = gdf.select(
    pl.len().alias("n_events"),
    pl.col("patient_id").n_unique().alias("n_patients"),
    pl.col("event_type").n_unique().alias("n_event_types"),
).collect(engine="streaming")
print("global_event_df stats:", stats.to_dicts()[0])

assert stats["n_events"][0] > 0
assert stats["n_patients"][0] > 0
assert stats["n_event_types"][0] >= 3

# Event-type distribution (top few)
et_counts = (
    gdf.group_by("event_type")
    .agg(pl.len().alias("n"))
    .sort("n", descending=True)
    .collect(engine="streaming")
)
print("event_type counts (top):")
print(et_counts.head(10))

# Assert critical event types exist.
expected_event_types = {"patients", "admissions", "icustays", "labevents", "chartevents"}
present = set(et_counts["event_type"].to_list())
missing = expected_event_types - present
assert not missing, f"Missing required event types: {missing}"

# Check timestamps parse: icustays/admissions should have non-null timestamps.
for et in ["icustays", "admissions", "labevents", "chartevents"]:
    non_null_ts = (
        gdf.filter(pl.col("event_type") == et)
        .select(pl.col("timestamp").is_not_null().sum().alias("non_null_ts"), pl.len().alias("n"))
        .collect(engine="streaming")
    )
    print(f"{et}: non_null_ts {non_null_ts['non_null_ts'][0]}/{non_null_ts['n'][0]}")
    assert non_null_ts["n"][0] > 0, f"No rows for event_type={et}"
    assert non_null_ts["non_null_ts"][0] > 0, f"All timestamps null for event_type={et}"

# Show a concrete icustays row and verify required attributes are present.
icu_sample = (
    gdf.filter(pl.col("event_type") == "icustays")
    .select(
        "patient_id",
        "timestamp",
        pl.col("icustays/hadm_id"),
        pl.col("icustays/stay_id"),
        pl.col("icustays/outtime"),
        pl.col("icustays/first_careunit"),
    )
    .head(3)
    .collect(engine="streaming")
)
print("icustays sample rows:")
print(icu_sample)
assert icu_sample.height > 0

# Verify lab join worked: labevents should have itemid/valuenum/label columns.
lab_sample = (
    gdf.filter(pl.col("event_type") == "labevents")
    .select(
        "patient_id",
        "timestamp",
        pl.col("labevents/itemid"),
        pl.col("labevents/valuenum"),
        pl.col("labevents/label"),
    )
    .head(5)
    .collect(engine="streaming")
)
print("labevents sample rows:")
print(lab_sample)
assert lab_sample.height > 0
assert lab_sample["labevents/itemid"].null_count() < lab_sample.height

# -------------------------------------------------------------------
# B2) Task + processors + model training smoke test
# -------------------------------------------------------------------
task = RemainingLengthOfStayTPC_MIMIC4(labevent_itemids=LAB_ITEMIDS, chartevent_itemids=[])

sd = mimic.set_task(task)
print("num samples:", len(sd))
assert len(sd) > 0, "Task produced 0 samples; check cohort filters + available events."

print("one sample keys:", sd[0].keys())
print(
    "ts shape:", tuple(sd[0]["ts"].shape),
    " static shape:", tuple(sd[0]["static"].shape),
    " y shape:", tuple(sd[0]["y"].shape),
)

# Basic processed tensor invariants.
assert sd[0]["ts"].dim() == 3 and sd[0]["ts"].shape[-1] == 2
assert sd[0]["y"].dim() == 1 and sd[0]["y"].shape[0] == sd[0]["ts"].shape[0]

train_ds, val_ds, test_ds = split_by_patient(sd, ratios=[0.8, 0.1, 0.1])
train_loader = get_dataloader(train_ds, batch_size=8, shuffle=True)
val_loader = get_dataloader(val_ds, batch_size=8, shuffle=False)
test_loader = get_dataloader(test_ds, batch_size=8, shuffle=False)

model2 = TPC(dataset=sd)
trainer = Trainer(model2, metrics=["mae", "mse"], enable_logging=False)
trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    test_dataloader=test_loader,
    epochs=1,
    monitor="mae",
    monitor_criterion="min",
)


Using default EHR config: /home/kevinchen101/PyHealth-TPC_LoS-AKV/pyhealth/datasets/configs/mimic4_ehr.yaml
Memory usage Before initializing mimic4_ehr: 2245.1 MB
Duplicate table names in tables list. Removing duplicates.
Initializing mimic4_ehr dataset from /home/kevinchen101/PyHealth-TPC_LoS-AKV/datasets/mimic-iv-demo/2.2 (dev mode: True)
Using provided cache_dir: /home/kevinchen101/PyHealth-TPC_LoS-AKV/.pyhealth_dataset_cache/b1ac8f11-4451-5717-b5db-18b569f2ffe5
Memory usage After initializing mimic4_ehr: 2245.1 MB
Found cached event dataframe: /home/kevinchen101/PyHealth-TPC_LoS-AKV/.pyhealth_dataset_cache/b1ac8f11-4451-5717-b5db-18b569f2ffe5/global_event_df.parquet
global_event_df stats: {'n_events': 113970, 'n_patients': 19, 'n_event_types': 5}
event_type counts (top):
shape: (5, 2)
┌─────────────┬───────┐
│ event_type  ┆ n     │
│ ---         ┆ ---   │
│ str         ┆ u32   │
╞═════════════╪═══════╡
│ chartevents ┆ 88556 │
│ labevents   ┆ 25309 │
│ admissions  ┆ 61    │
│ icusta

Epoch 0 / 1:   0%|          | 0/2 [00:00<?, ?it/s]

--- Train epoch-0, step-2 ---
loss: 1.6347


Evaluation: 100%|██████████| 1/1 [00:00<00:00, 103.10it/s]

--- Eval epoch-0, step-2 ---
mae: 0.8940
mse: 0.8732
loss: 2.4583
New best mae score (0.8940) at epoch-0, step-2



Evaluation: 100%|██████████| 1/1 [00:00<00:00, 96.64it/s]

--- Test ---
mae: 2.3715
mse: 12.6283
loss: 2.0112
